In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,root_mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 43

In [ ]:
df = pd.read_csv('data/games_list.csv')
# df = df.sort_values(by='avg_concurrent_players_after_90',ascending=True).iloc[10:]
df.head(10)


In [ ]:

df['log_estimated_launch_copies'] = np.log1p(df['estimated_launch_copies_sold'])
df['log_estimated_launch_reviews'] = np.log1p(df['estimated_launch_reviews'])
df['log_estimated_launch_followers'] = np.log1p(df['estimated_launch_followers'])
df['log_avg_playtime'] = np.log1p(df['avg_playtime'])
df['log_review_score'] = np.log1p(df['review_score'])
df['log_avg_concurrent_players_after_90'] = np.log1p(df['avg_concurrent_players_after_90'])


# Log calculations 
df['log_review_to_follower_ratio'] = df['log_estimated_launch_reviews'] - df['log_estimated_launch_followers']
df['log_copies_to_follower_ratio'] = df['log_estimated_launch_copies'] - df['log_estimated_launch_followers']
# ratios for logs are subtraction due to the rules of logs log(a/b) = log(a) - log(b)
df['log_engagement_quality'] = df['log_avg_playtime'] + df['log_review_score']
# log (a*b) = log(a) + log(b)
df['log_virality_factor'] = df['log_review_to_follower_ratio'] + df['log_review_score']

# features we want to train on
features = [
    'log_estimated_launch_copies',
    'log_review_to_follower_ratio',
    'log_copies_to_follower_ratio',
    'log_engagement_quality',
    'log_virality_factor',
    'log_review_score',
    'avg_sentiment'
]





In [ ]:
X = df[features]
y = df['log_avg_concurrent_players_after_90']
X.head(1)

In [ ]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE)

In [ ]:
# standard random forest without any settings changed
random_forest_base = RandomForestRegressor()
random_forest_base.fit(X_train,y_train)

In [ ]:
y_pred_log = random_forest_base.predict(X_test)

In [ ]:
mean_absolute_error(y_test, y_pred_log)

In [ ]:
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mean_absolute_error(y_test_orig,y_pred_orig)

In [ ]:
root_mean_squared_error(y_test_orig,y_pred_orig)

In [ ]:
r2_score(y_test_orig,y_pred_orig)

In [ ]:
params= {
    'n_estimators': [200, 500],
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [1, 2, 5]
}

In [ ]:
estimator = RandomForestRegressor(n_estimators=100,max_features=1/3,oob_score=True, random_state=RANDOM_STATE)
random_forest_cv = GridSearchCV(estimator=estimator,param_grid=params,cv=5,scoring='neg_root_mean_squared_error',n_jobs=-1)

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
random_forest_cv.fit(X_train,y_train)

In [ ]:
y_pred_log = random_forest_cv.predict(X_test)

In [ ]:
mean_absolute_error(y_test, y_pred_log)

In [ ]:
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mean_absolute_error(y_test_orig,y_pred_orig)

In [ ]:
r2_score(y_test_orig,y_pred_orig)

In [ ]:
root_mean_squared_error(y_pred=y_pred_orig,y_true=y_test_orig)

In [ ]:
X.head(1)

# New Data Test
This code below is used for testing new games that aren't in our data

In [ ]:
from add_new_games import fetch_gamalytic_data, fetch_game,get_sentiment_data

user_input = str(input('Enter name of game'))
appid = fetch_game(user_input)
new_game = fetch_gamalytic_data(appid=appid)
new_game = get_sentiment_data(new_game)


# Convert to DataFrame
df = pd.DataFrame([new_game])

# Log-transform base features
df['log_estimated_launch_reviews'] = np.log1p(df['estimated_launch_reviews'])
df['log_estimated_launch_followers'] = np.log1p(df['estimated_launch_followers'])
df['log_estimated_launch_copies'] = np.log1p(df['estimated_launch_copies_sold'])
df['log_avg_playtime'] = np.log1p(df['avg_playtime'])
df['log_review_score'] = np.log1p(df['review_score'])

# Engineer features (same as training)
df['log_review_to_follower_ratio'] = df['log_estimated_launch_reviews'] - df['log_estimated_launch_followers']
df['log_copies_to_follower_ratio'] = df['log_estimated_launch_copies'] - df['log_estimated_launch_followers']
df['log_engagement_quality'] = df['log_avg_playtime'] + df['log_review_score']
df['log_virality_factor'] = df['log_review_to_follower_ratio'] + df['log_review_score']

# Select features (same order as training)
features = [
    'log_estimated_launch_copies',
    'log_review_to_follower_ratio',
    'log_copies_to_follower_ratio',
    'log_engagement_quality',
    'log_virality_factor',
    'log_review_score',
    'avg_sentiment'
]

X_new = df[features]

# Predict
y_pred_log = random_forest_cv.predict(X_new)
y_pred_original = np.expm1(y_pred_log)

print(f"\n{'='*60}")
print(f"Prediction for: {new_game['name']}")
print(f"{'='*60}")
print(f"Expected avg concurrent players (90 days): {y_pred_original[0]:,.0f}")
print(f"{'='*60}")


In [ ]:
importances = random_forest_cv.best_estimator_.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values(by='importance', ascending=False)


print(feature_importance_df)

# Optional: plot
plt.figure(figsize=(8,5))
plt.barh(feature_importance_df['feature'], feature_importance_df['importance'])
plt.xlabel("Importance")
plt.title("Random Forest Feature Importances")
plt.gca().invert_yaxis()
plt.show()